# HS4002 Week 8

## Practising with Count Models

Research question: **what factors influence the number of children someone has?**

We'll compare three model types for count outcomes:
1. OLS (baseline)
2. Poisson regression
3. Negative Binomial regression

Plus: overdispersion testing and marginal effects.

In [1]:
# !pip install pandas numpy plotnine statsmodels stargazer marginaleffects scipy

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [2]:
raw_df = pd.read_csv("gss2022_mini2.csv")

chosen_variables = [
	'id', 'age', 'happy', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat', 'biokids', 'educ'
]
df = raw_df[chosen_variables].dropna().copy()

income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont']   = df['income16'].map(income_map)
df['income_10k']   = df['income_cont'] / 10000
df['age']       = raw_df.loc[df.index, 'age'] - 18

df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

df['race_bin']   = pd.Categorical(df['race'].astype(str))
df['race_cat']   = df['race'].map({1: 'White', 2: 'Black'}).fillna('Other')
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = pd.Categorical(
    np.where(df['sex'] == 2, 'Woman', 'Not Woman'),
    categories=['Not Woman', 'Woman']
)
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')
df['happy_bin']  = (df['happy'] <= 2).astype(int)
df['happy_cat']  = np.where(df['happy'] <= 2, 'Happy', 'Not Happy')

# Starting with an OLS model

Regress no. of children (`biokids`) against income in ten thousands (`income_10k`). Note: one unit = $10,000. Name this `ols_lm1`.

In [3]:
ols_lm1 = smf.ols('biokids ~ income_10k', data=df).fit()
print(ols_lm1.summary())

                            OLS Regression Results                            
Dep. Variable:                biokids   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                   0.04228
Date:                Wed, 07 Oct 2026   Prob (F-statistic):              0.837
Time:                        14:15:37   Log-Likelihood:                -1050.5
No. Observations:                 557   AIC:                             2105.
Df Residuals:                     555   BIC:                             2114.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.5647      0.107     14.639      0.0

How would you interpret the coefficient for `income_10k`? What is unusual about using OLS on a count variable?

# Continuing with a Poisson Model

Use `smf.poisson()` — equivalent to `glm(..., family='poisson')` in R.

In [4]:
po_lm1 = smf.poisson('biokids ~ income_10k', data=df).fit()
print(po_lm1.summary())

Optimization terminated successfully.
         Current function value: 1.767601
         Iterations 3
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      555
Method:                           MLE   Df Model:                            1
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:               3.454e-05
Time:                        14:16:01   Log-Likelihood:                -984.55
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                    0.7942
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4478      0.053      8.410      0.000       0.343       0.552
income_10k     0.0012      0.

In [5]:
# Interpretation of income_10k. 
# The coefficient of income_10k is 0.0012, therefore: a one unit change in income_10k is expected to change the number of children by a factor of exp(0.0012), ceteris paribus
# Alternatively, it can be a percentage change of : 100 x [exp(0.0012)-1]% 

Interpret the coefficient on `income_10k`. Do not read it as you would an OLS coefficient.

Poisson coefficients are **factors of change**: a one-unit change in `income_10k` is expected to change the number of children by a factor of exp(β), ceteris paribus.

You can alternatively read this as a **percentage change**: 100 × [exp(β) − 1]%.

In [6]:
import numpy as np
b = po_lm1.params['income_10k']
print(f'Factor of change, exp(b): {np.exp(b):.4f}')
print(f'Percentage change, 100 * [exp(b) - 1]: {100 * (np.exp(b) - 1):.2f}%')

Factor of change, exp(b): 1.0012
Percentage change, 100 * [exp(b) - 1]: 0.12%


# Adding Age

Add `age` to both models. `age` is centered at 18, so 0 is an 18-year-old. Save as `ols_lm2` and `po_lm2`.

In [8]:
ols_lm2 = smf.ols('biokids ~ income_10k + age', data=df).fit()
po_lm2  = smf.poisson('biokids ~ income_10k + age', data=df).fit()
print(po_lm2.summary())

Optimization terminated successfully.
         Current function value: 1.720780
         Iterations 5
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      554
Method:                           MLE   Df Model:                            2
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                 0.02652
Time:                        14:25:27   Log-Likelihood:                -958.47
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                 4.563e-12
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0674      0.094     -0.720      0.472      -0.251       0.116
income_10k     0.0008      0.

## Comparing the Poisson Models via Likelihood Ratio Test

In [9]:
lr_stat = 2 * (po_lm2.llf - po_lm1.llf)
df_diff = po_lm2.df_model - po_lm1.df_model
p_val   = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}, df = {df_diff:.0f}, p-value = {p_val:.6f}')

LR statistic = 52.1579, df = 1, p-value = 0.000000


In [10]:
# A likelihood ratio test indicated that adding age significantly improved the fit of the Poisson regression model, LR(1) = 52.16, p< 0.001, this suggests
# Suggests that the model including age fits the data substantially better than the model containing income alone

# Building a Negative Binomial Model

Negative binomial relaxes the Poisson assumption that variance = mean (allowing overdispersion).

Equivalent to R's `MASS::glm.nb()`.

Read the coefficients the same way as in a Poisson model: as a factor of change, exp(β), or as a percentage change, 100 × [exp(β) − 1]%.

In [11]:
nb_m1 = smf.negativebinomial('biokids ~ income_10k + age', data=df).fit()
print(nb_m1.summary())

Optimization terminated successfully.
         Current function value: 1.670251
         Iterations: 14
         Function evaluations: 19
         Gradient evaluations: 19
                     NegativeBinomial Regression Results                      
Dep. Variable:                biokids   No. Observations:                  557
Model:               NegativeBinomial   Df Residuals:                      554
Method:                           MLE   Df Model:                            2
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                 0.01718
Time:                        14:31:16   Log-Likelihood:                -930.33
converged:                       True   LL-Null:                       -946.59
Covariance Type:            nonrobust   LLR p-value:                 8.681e-08
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0832      0.116     -

In [12]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_lm2, po_lm2, nb_m1])
sg.title('Number of Children')
sg.custom_columns(['OLS', 'Poisson', 'Neg. Binomial'], [1, 1, 1])

# Ending the cell on the object itself renders the table in the notebook
sg

## Test for Overdispersion

Was NB necessary? We test whether the NB's dispersion parameter (alpha) is significantly different from 0. If alpha = 0, the NB reduces to Poisson.

This is equivalent to R's `pscl::odTest()`.

In [13]:
# Overdispersion test: LR comparing NB vs Poisson
# Under H0 (Poisson), the test statistic has a chi-square(1)/2 distribution
# because the null is on the boundary of the parameter space
lr_od = 2 * (nb_m1.llf - po_lm2.llf)
p_od  = 0.5 * chi2.sf(lr_od, 1)
print(f'Overdispersion LR statistic = {lr_od:.4f}')
print(f'p-value (one-sided) = {p_od:.6f}')
print('NB preferred over Poisson:', p_od < 0.05)

Overdispersion LR statistic = 56.2895
p-value (one-sided) = 0.000000
NB preferred over Poisson: True


# Marginal Effects

A marginal effect is the expected change in the number of children when an independent variable increases by one unit, ceteris paribus. Compute them the same way as for the logit models.

In [14]:
from marginaleffects import avg_comparisons, comparisons

## Average marginal effects of income

In [15]:
avg_comparisons(po_lm2, variables='income_10k')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.001238,0.007496,0.165145,0.86889,0.202754,-0.013487,0.015962


In [16]:
avg_comparisons(nb_m1, variables='income_10k')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.00077,0.009532,0.080801,0.93563,0.095991,-0.017952,0.019493


## Marginal effect at the mean of income

In [17]:
comparisons(po_lm2, variables='income_10k', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.001201,0.007274,0.165154,0.868883,0.202767,-0.013086,0.015488


In [18]:
comparisons(nb_m1, variables='income_10k', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.000746,0.009229,0.080801,0.935629,0.095991,-0.017383,0.018874


# Independent Investigation: Education and Number of Children

Investigate the relationship between **no. of children** (`biokids`) and **BA education** (`ba_binary`).

Do the following:

1. Include these independent variables: `ba_binary`, `age`, `income_10k`, `sex_woman`, and `race_bin`.
2. Build OLS, Poisson, and Negative Binomial models.
3. Produce a Stargazer table comparing the three models.
4. Compute marginal effects for each model.
5. Decide which model is best. Justify.
6. Answer: does a person with a BA have more or fewer children, and by how much? For the Poisson and negative binomial models, report a factor of change or a percentage change.

Poisson Regression Results, OLS, Negative Binom


In [21]:
ols_lm3 = smf.ols('biokids ~ income_10k + age + ba_binary + sex_woman + race_bin', data=df).fit()
po_lm3  = smf.poisson('biokids ~ income_10k + age + ba_binary + sex_woman + race_bin', data=df).fit()
print(po_lm3.summary())
print(ols_lm3.summary())

Optimization terminated successfully.
         Current function value: 1.671346
         Iterations 5
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      550
Method:                           MLE   Df Model:                            6
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                 0.05449
Time:                        15:23:35   Log-Likelihood:                -930.94
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                 7.505e-21
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.3234      0.112     -2.885      0.004      -0.543      -0.104
sex_w

In [20]:
nb_m2 = smf.negativebinomial('biokids ~ income_10k + age +  ba_binary + sex_woman + race_bin', data=df).fit()
print(nb_m2.summary())

Optimization terminated successfully.
         Current function value: 1.637030
         Iterations: 28
         Function evaluations: 34
         Gradient evaluations: 34
                     NegativeBinomial Regression Results                      
Dep. Variable:                biokids   No. Observations:                  557
Model:               NegativeBinomial   Df Residuals:                      550
Method:                           MLE   Df Model:                            6
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                 0.03673
Time:                        15:22:54   Log-Likelihood:                -911.83
converged:                       True   LL-Null:                       -946.59
Covariance Type:            nonrobust   LLR p-value:                 5.111e-13
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             

Stargazer table

In [22]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_lm3, po_lm3, nb_m2])
sg.title('Number of Children')
sg.custom_columns(['OLS', 'Poisson', 'Neg. Binomial'], [1, 1, 1])

# Ending the cell on the object itself renders the table in the notebook
sg

In [23]:
avg_comparisons(po_lm3)

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""age""","""+1""",0.024414,0.003355,7.277538,1.1797e-12,39.624689,0.017824,0.031004
"""ba_binary""","""1 - 0""",-0.485716,0.116509,-4.168927,0.000036,14.779928,-0.714573,-0.25686
"""income_10k""","""+1""",0.022427,0.008192,2.737725,0.006387,7.29067,0.006336,0.038519
"""race_bin""","""2.0 - 1.0""",0.673268,0.194387,3.463541,0.000575,10.765068,0.291436,1.0551
"""race_bin""","""3.0 - 1.0""",0.059596,0.243549,0.244699,0.806781,0.309751,-0.418804,0.537996
"""sex_woman""","""Woman - Not Woman""",0.495847,0.107462,4.614151,0.000005,17.634074,0.28476,0.706933


In [24]:
avg_comparisons(ols_lm3)

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""age""","""+1""",0.023289,0.003755,6.201883,1.0970e-9,29.763832,0.015912,0.030665
"""ba_binary""","""1 - 0""",-0.481404,0.141182,-3.409812,0.000698,10.484971,-0.758726,-0.204082
"""income_10k""","""+1""",0.021642,0.009994,2.165448,0.030783,5.021745,0.00201,0.041274
"""race_bin""","""2.0 - 1.0""",0.626281,0.200915,3.117148,0.001922,9.02354,0.231627,1.020935
"""race_bin""","""3.0 - 1.0""",0.064282,0.281902,0.228029,0.819708,0.286817,-0.489454,0.618018
"""sex_woman""","""Woman - Not Woman""",0.493528,0.129934,3.79829,0.000162,12.592322,0.2383,0.748756


In [25]:
avg_comparisons(nb_m2)

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""age""","""+1""",0.025356,0.004225,6.002005,3.5394e-9,28.073832,0.017058,0.033655
"""ba_binary""","""1 - 0""",-0.515433,0.145226,-3.54919,0.000419,11.219114,-0.800698,-0.230169
"""income_10k""","""+1""",0.023667,0.010277,2.302987,0.021652,5.529366,0.003481,0.043854
"""race_bin""","""2.0 - 1.0""",0.683987,0.25011,2.73474,0.006444,7.27776,0.192698,1.175276
"""race_bin""","""3.0 - 1.0""",0.063408,0.295497,0.21458,0.830174,0.268513,-0.517033,0.643849
"""sex_woman""","""Woman - Not Woman""",0.505913,0.132581,3.815875,0.000151,12.69222,0.245486,0.766341


Having a BA is associated with about 0.48 to 0.52 fewer children on average. All three models agree, and all are significant (p < 0.001).

In [27]:
# If NB was necessary 

lr_od = 2 * (nb_m2.llf - po_lm3.llf)
p_od  = 0.5 * chi2.sf(lr_od, 1)
print(f'Overdispersion LR statistic = {lr_od:.4f}')
print(f'p-value (one-sided) = {p_od:.6f}')
print('NB preferred over Poisson:', p_od < 0.05)

Overdispersion LR statistic = 38.2281
p-value (one-sided) = 0.000000
NB preferred over Poisson: True


Comparing AIC, BIC


In [30]:
for name, m in [('OLS', ols_lm3), ('Poisson', po_lm3), ('NB', nb_m2)]:
    print(f'{name}: AIC = {m.aic:.1f}, BIC = {m.bic:.1f}')

OLS: AIC = 2044.9, BIC = 2075.2
Poisson: AIC = 1875.9, BIC = 1906.1
NB: AIC = 1839.7, BIC = 1874.2


From the above 3 models, 
 NB is the best model: the overdispersion test is significant (p < 0.001), and it has the lowest AIC (1839.7) and BIC (1874.2).
# Poisson assumes variance = mean, so its standard errors are too small; NB corrects this.
# OLS is unsuitable for counts: it can predict negative values and assumes constant variance.


A person with a BA is expected to have fewer children, holding age, income, sex and race constant. In the negative binomial model, which is the best model, having a BA changes the expected number of children by a factor of 0.72, about 28% fewer. The Poisson model gives a similar result, a factor of 0.73 or about 27% fewer. In absolute terms, that is about half a child fewer on average (marginal effects of −0.48 for OLS, −0.49 for Poisson and −0.52 for NB). The effect is statistically significant in all three models (p < 0.001).